# DC clean sub-08 generation

流程：直接写路径 -> 用你的 ATMS `40.pth` 提取 EEG embedding -> 加载你已有的 150 epochs diffusion prior -> SDXL/IP-Adapter 生成图片。

## 1. 路径和运行参数

In [ ]:
import os
import sys

# 如果模型还没有缓存，国内服务器建议保留这几行；如果都已缓存，不影响直接使用缓存。
os.environ.setdefault("HF_ENDPOINT", "https://hf-mirror.com")
os.environ.setdefault("HF_HOME", "/data/gaoy/projects/.cache/huggingface")
os.environ.setdefault("HUGGINGFACE_HUB_CACHE", "/data/gaoy/projects/.cache/huggingface/hub")
os.environ.setdefault("TORCH_HOME", "/data/gaoy/.cache/torch")

REPO_ROOT = "/data/gaoy/projects/EEG_Image_decode"
GENERATION_DIR = "/data/gaoy/projects/EEG_Image_decode/Generation"
DATA_ROOT = "/data/gaoy/projects/datasets/EEG_Image_decode"

SUBJECT = "sub-08"
SUBJECT_ID = 8
DEVICE = "cuda:0"
SEED = 42

EEG_DATA_DIR = f"{DATA_ROOT}/Preprocessed_data_250Hz/{SUBJECT}"
TRAIN_IMAGE_DIR = f"{DATA_ROOT}/images_set/training_images"
TEST_IMAGE_DIR = f"{DATA_ROOT}/images_set/test_images"
VIT_TRAIN_FEATURES = f"{DATA_ROOT}/ViT-H-14_features_train.pt"
VIT_TEST_FEATURES = f"{DATA_ROOT}/ViT-H-14_features_test.pt"

# 这里改成你自己的 ATMS 40.pth。
ATMS_CKPT = "/data/gaoy/projects/EEG_Image_decode/models/contrast/ATMS/sub-08/08-18_08-11/40.pth"

# 这里改成你已经训练好的 150 epochs diffusion prior。
DIFFUSION_PRIOR_CKPT = "/data/gaoy/projects/datasets/EEG_Image_decode/fintune_ckpts/sub-08/diffusion_prior_my_150epochs.pt"

# 生成图片输出目录。
OUTPUT_DIR = "/data/gaoy/projects/datasets/EEG_Image_decode/generated_imgs/sub-08_my_150epochs"

START_INDEX = 0
NUM_CONCEPTS = 200
REPEATS_PER_CONCEPT = 10
ATMS_BATCH_SIZE = 1024

for p in [REPO_ROOT, GENERATION_DIR]:
    if p not in sys.path:
        sys.path.insert(0, p)

for p in [EEG_DATA_DIR, TRAIN_IMAGE_DIR, TEST_IMAGE_DIR, VIT_TRAIN_FEATURES, VIT_TEST_FEATURES, ATMS_CKPT]:
    assert os.path.exists(p), p

if os.path.exists(DIFFUSION_PRIOR_CKPT):
    print("found diffusion prior:", DIFFUSION_PRIOR_CKPT)
else:
    print("diffusion prior not found; train it in block 6:", DIFFUSION_PRIOR_CKPT)

print("output:", OUTPUT_DIR)

## 2. 导入库并固定随机种子

In [ ]:
import random
import numpy as np
import torch
import torch.nn as nn
from torch import Tensor
from torch.utils.data import Dataset, DataLoader
from einops.layers.torch import Rearrange

from models.subject_layers.Transformer_EncDec import Encoder, EncoderLayer
from models.subject_layers.SelfAttention_Family import FullAttention, AttentionLayer
from models.subject_layers.Embed import DataEmbedding
from models.loss import ClipLoss

from diffusion_prior import EmbeddingDataset, DiffusionPriorUNet, Pipe
from custom_pipeline import Generator4Embeds

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)
device = torch.device(DEVICE if torch.cuda.is_available() else "cpu")
print("device:", device)
print("cuda count:", torch.cuda.device_count())

## 3. 读取 concept 名称、CLIP 图像特征和 EEG 数据

In [ ]:
def load_torch_features(path, key):
    obj = torch.load(path, map_location="cpu")
    return obj[key] if isinstance(obj, dict) else obj

def load_npy_dict(path):
    obj = np.load(path, allow_pickle=True)
    if isinstance(obj, np.ndarray) and obj.shape == ():
        obj = obj.item()
    return obj

def select_time_window(eeg, times, start=0.0, end=1.0):
    times = torch.as_tensor(times).float()
    if eeg.shape[-1] != len(times) and eeg.shape[-1] == len(times[50:]):
        times = times[50:]
    if eeg.shape[-1] != len(times):
        return eeg
    keep = (times >= start) & (times <= end)
    return eeg[..., keep]

def image_files(folder):
    return sorted([
        f for f in os.listdir(folder)
        if f.lower().endswith((".png", ".jpg", ".jpeg"))
    ])

def collect_image_paths(root):
    folders = [d for d in os.listdir(root) if os.path.isdir(os.path.join(root, d))]
    folders.sort()
    names = [folder[folder.index("_") + 1:] if "_" in folder else folder for folder in folders]
    paths = []
    for folder in folders:
        folder_path = os.path.join(root, folder)
        for image_name in image_files(folder_path):
            paths.append(os.path.join(folder_path, image_name))
    return folders, names, paths

def load_eeg_split(train):
    filename = "preprocessed_eeg_training.npy" if train else "preprocessed_eeg_test.npy"
    data = load_npy_dict(os.path.join(EEG_DATA_DIR, filename))
    eeg = torch.from_numpy(data["preprocessed_eeg_data"]).float()
    eeg = select_time_window(eeg, data["times"])

    if train:
        # Match the author's explicit order: class -> 10 images -> 4 EEG repetitions.
        assert eeg.ndim == 4, f"Expected training EEG [16540, 4, 63, 250], got {tuple(eeg.shape)}"
        assert eeg.shape[0] == 1654 * 10 and eeg.shape[1] == 4, tuple(eeg.shape)
        data_list = []
        label_list = []
        for class_idx in range(1654):
            start = class_idx * 10
            block = eeg[start:start + 10]
            data_list.append(block)
            label_list.append(torch.full((10,), class_idx, dtype=torch.long))
        eeg = torch.cat(data_list, dim=0).view(-1, eeg.shape[-2], eeg.shape[-1])
        labels = torch.cat(label_list, dim=0).repeat_interleave(4)
    else:
        # Match the author's test order: class -> average 80 trials -> one EEG sample.
        assert eeg.ndim == 4, f"Expected test EEG [200, 80, 63, 250], got {tuple(eeg.shape)}"
        assert eeg.shape[0] == 200, tuple(eeg.shape)
        data_list = []
        label_list = []
        for class_idx in range(200):
            data_list.append(torch.mean(eeg[class_idx:class_idx + 1].squeeze(0), dim=0))
            label_list.append(torch.tensor([class_idx], dtype=torch.long))
        eeg = torch.cat(data_list, dim=0).view(-1, eeg.shape[-2], eeg.shape[-1])
        labels = torch.cat(label_list, dim=0)

    return eeg, labels

train_folders, train_concepts, train_image_paths = collect_image_paths(TRAIN_IMAGE_DIR)
test_folders, texts, test_image_paths = collect_image_paths(TEST_IMAGE_DIR)
emb_img_train = load_torch_features(VIT_TRAIN_FEATURES, "img_features").float()
emb_img_test = load_torch_features(VIT_TEST_FEATURES, "img_features").float()
train_eeg, train_labels = load_eeg_split(train=True)
test_eeg, test_labels = load_eeg_split(train=False)

assert len(train_folders) == 1654, len(train_folders)
assert len(test_folders) == 200, len(test_folders)
assert len(train_image_paths) == 1654 * 10, len(train_image_paths)
assert len(test_image_paths) == 200, len(test_image_paths)
assert len(set(texts)) == len(texts), "Duplicate test concept names after stripping folder prefix."

print("train concepts/images:", len(train_folders), len(train_image_paths))
print("test concepts/images:", len(texts), len(test_image_paths))
print("emb_img_train:", emb_img_train.shape)
print("emb_img_test:", emb_img_test.shape)
print("train_eeg:", train_eeg.shape, "train_labels:", train_labels.shape)
print("test_eeg:", test_eeg.shape, "test_labels:", test_labels.shape)

## 4. 最小 ATMS 定义

In [ ]:
class Config:
    def __init__(self):
        self.task_name = "classification"
        self.seq_len = 250
        self.pred_len = 250
        self.output_attention = False
        self.d_model = 250
        self.embed = "timeF"
        self.freq = "h"
        self.dropout = 0.25
        self.factor = 1
        self.n_heads = 4
        self.e_layers = 1
        self.d_ff = 256
        self.activation = "gelu"
        self.enc_in = 63

class iTransformer(nn.Module):
    def __init__(self, configs):
        super().__init__()
        self.enc_embedding = DataEmbedding(
            configs.seq_len, configs.d_model, configs.embed, configs.freq,
            configs.dropout, joint_train=False, num_subjects=10,
        )
        self.encoder = Encoder(
            [EncoderLayer(
                AttentionLayer(
                    FullAttention(False, configs.factor, attention_dropout=configs.dropout, output_attention=configs.output_attention),
                    configs.d_model, configs.n_heads,
                ),
                configs.d_model,
                configs.d_ff,
                dropout=configs.dropout,
                activation=configs.activation,
            ) for _ in range(configs.e_layers)],
            norm_layer=torch.nn.LayerNorm(configs.d_model),
        )

    def forward(self, x_enc, x_mark_enc, subject_ids=None):
        enc_out = self.enc_embedding(x_enc, x_mark_enc, subject_ids)
        enc_out, _ = self.encoder(enc_out, attn_mask=None)
        return enc_out[:, :63, :]

class PatchEmbedding(nn.Module):
    def __init__(self, emb_size=40):
        super().__init__()
        self.tsconv = nn.Sequential(
            nn.Conv2d(1, 40, (1, 25), stride=(1, 1)),
            nn.AvgPool2d((1, 51), (1, 5)),
            nn.BatchNorm2d(40),
            nn.ELU(),
            nn.Conv2d(40, 40, (63, 1), stride=(1, 1)),
            nn.BatchNorm2d(40),
            nn.ELU(),
            nn.Dropout(0.5),
        )
        self.projection = nn.Sequential(
            nn.Conv2d(40, emb_size, (1, 1), stride=(1, 1)),
            Rearrange("b e (h) (w) -> b (h w) e"),
        )

    def forward(self, x: Tensor) -> Tensor:
        return self.projection(self.tsconv(x.unsqueeze(1)))

class ResidualAdd(nn.Module):
    def __init__(self, fn):
        super().__init__()
        self.fn = fn

    def forward(self, x, **kwargs):
        return self.fn(x, **kwargs) + x

class FlattenHead(nn.Module):
    def forward(self, x):
        return x.contiguous().view(x.size(0), -1)

class EncEeg(nn.Sequential):
    def __init__(self, emb_size=40):
        super().__init__(PatchEmbedding(emb_size), FlattenHead())

class ProjEeg(nn.Sequential):
    def __init__(self, embedding_dim=1440, proj_dim=1024, drop_proj=0.5):
        super().__init__(
            nn.Linear(embedding_dim, proj_dim),
            ResidualAdd(nn.Sequential(nn.GELU(), nn.Linear(proj_dim, proj_dim), nn.Dropout(drop_proj))),
            nn.LayerNorm(proj_dim),
        )

class ATMS(nn.Module):
    def __init__(self, num_channels=63, sequence_length=250, num_subjects=2):
        super().__init__()
        default_config = Config()
        self.encoder = iTransformer(default_config)
        self.subject_wise_linear = nn.ModuleList([nn.Linear(default_config.d_model, sequence_length) for _ in range(num_subjects)])
        self.enc_eeg = EncEeg()
        self.proj_eeg = ProjEeg()
        self.logit_scale = nn.Parameter(torch.ones([]) * np.log(1 / 0.07))
        self.loss_func = ClipLoss()

    def forward(self, x, subject_ids):
        x = self.encoder(x, None, subject_ids)
        return self.proj_eeg(self.enc_eeg(x))

## 5. 用你的 40.pth 提取 train/test EEG embedding

In [ ]:
class EEGTensorDataset(Dataset):
    def __init__(self, eeg, labels):
        self.eeg = eeg
        self.labels = labels

    def __len__(self):
        return self.eeg.shape[0]

    def __getitem__(self, idx):
        return self.eeg[idx], self.labels[idx]

@torch.no_grad()
def extract_embeddings(eeg_model, eeg, labels, image_bank, batch_size=1024):
    loader = DataLoader(EEGTensorDataset(eeg, labels), batch_size=batch_size, shuffle=False, num_workers=0)
    image_bank = image_bank.to(device).float()
    all_features = []
    correct = 0
    total = 0

    eeg_model.eval()
    for eeg_batch, label_batch in loader:
        eeg_batch = eeg_batch.to(device)
        label_batch = label_batch.to(device)
        subject_ids = torch.full((eeg_batch.size(0),), SUBJECT_ID, dtype=torch.long, device=device)
        eeg_features = eeg_model(eeg_batch, subject_ids).float()
        all_features.append(eeg_features.cpu())

        logits = eeg_model.logit_scale * eeg_features @ image_bank.T
        pred = torch.argmax(logits, dim=1)
        correct += (pred == label_batch).sum().item()
        total += label_batch.numel()

    return torch.cat(all_features, dim=0), correct / total

eeg_model = ATMS(63, 250, num_subjects=2)
eeg_model.load_state_dict(torch.load(ATMS_CKPT, map_location="cpu"))
eeg_model = eeg_model.to(device)
print("ATMS parameters:", sum(p.numel() for p in eeg_model.parameters() if p.requires_grad))

train_image_bank = emb_img_train[::10]
eeg_features_train, train_top1 = extract_embeddings(eeg_model, train_eeg, train_labels, train_image_bank, ATMS_BATCH_SIZE)
eeg_features_test, test_top1 = extract_embeddings(eeg_model, test_eeg, test_labels, emb_img_test, ATMS_BATCH_SIZE)

print("eeg_features_train:", eeg_features_train.shape, "train top1:", round(train_top1, 4))
print("eeg_features_test:", eeg_features_test.shape, "test top1:", round(test_top1, 4))

## 6. 检查 diffusion prior 训练配对


In [ ]:
emb_img_train_by_author_repeat = emb_img_train.view(1654, 10, 1, 1024).repeat(1, 1, 4, 1).view(-1, 1024)
emb_img_train_4 = emb_img_train.view(1654, 10, 1024).repeat_interleave(4, dim=1).view(-1, 1024)
assert torch.allclose(emb_img_train_4, emb_img_train_by_author_repeat), "repeat_interleave does not match author repeat."
assert eeg_features_train.shape[0] == emb_img_train_4.shape[0] == train_eeg.shape[0]

def print_pairing_debug(n=20):
    print("TRAIN EEG -> image/CLIP pairing")
    for sample_idx in range(min(n, len(train_labels))):
        image_idx = sample_idx // 4
        rep_idx = sample_idx % 4
        class_idx = image_idx // 10
        image_in_class = image_idx % 10
        label = int(train_labels[sample_idx])
        assert label == class_idx, (sample_idx, label, class_idx)
        assert torch.allclose(emb_img_train_4[sample_idx], emb_img_train[image_idx]), sample_idx
        print(
            f"sample={sample_idx:03d} label={label:04d} image_idx={image_idx:04d} "
            f"image_in_class={image_in_class} rep={rep_idx} path={train_image_paths[image_idx]}"
        )

    print("\nTEST EEG -> concept / generated / ground-truth pairing")
    for sample_idx in range(min(n, len(test_labels))):
        label = int(test_labels[sample_idx])
        assert label == sample_idx, (sample_idx, label)
        print(
            f"sample={sample_idx:03d} label={label:03d} concept={texts[sample_idx]} "
            f"gt={test_image_paths[sample_idx]} generated={os.path.join(OUTPUT_DIR, texts[sample_idx], '0.png')}"
        )

print_pairing_debug(20)


## 7. 初始化 diffusion prior


In [ ]:
diffusion_prior = DiffusionPriorUNet(cond_dim=1024, dropout=0.1)
print("diffusion prior parameters:", sum(p.numel() for p in diffusion_prior.parameters() if p.requires_grad))
pipe = Pipe(diffusion_prior, device=device)


## 8. 训练 diffusion prior（默认注释，不运行）


In [ ]:
# 默认不训练。要重新训练 diffusion prior 时，取消本 block 的注释。
# 训练时建议先不要运行后面的推理 block，确认保存成功后再推理。
# train_dataset = EmbeddingDataset(c_embeddings=eeg_features_train, h_embeddings=emb_img_train_4)
# train_loader = DataLoader(train_dataset, batch_size=1024, shuffle=True, num_workers=8)
# pipe.train(train_loader, num_epochs=150, learning_rate=1e-3)
# os.makedirs(os.path.dirname(DIFFUSION_PRIOR_CKPT), exist_ok=True)
# torch.save(pipe.diffusion_prior.state_dict(), DIFFUSION_PRIOR_CKPT)
# print("saved diffusion prior:", DIFFUSION_PRIOR_CKPT)


## 9. 推理生成图片（默认运行）


In [ ]:
# 默认推理。要只训练 diffusion prior 时，可以不运行这个 block 或先整块注释掉。
assert os.path.exists(DIFFUSION_PRIOR_CKPT), DIFFUSION_PRIOR_CKPT
pipe.diffusion_prior.load_state_dict(torch.load(DIFFUSION_PRIOR_CKPT, map_location=device))
pipe.diffusion_prior.eval()
print("loaded diffusion prior:", DIFFUSION_PRIOR_CKPT)

os.makedirs(OUTPUT_DIR, exist_ok=True)
generator = Generator4Embeds(num_inference_steps=4, device=device)

end_index = min(START_INDEX + NUM_CONCEPTS, len(texts), eeg_features_test.shape[0])
for k in range(START_INDEX, end_index):
    prior_generator = torch.Generator(device=device).manual_seed(SEED + k)
    eeg_embeds = eeg_features_test[k:k + 1].to(device)
    h = pipe.generate(c_embeds=eeg_embeds, num_inference_steps=50, guidance_scale=5.0, generator=prior_generator)

    for j in range(REPEATS_PER_CONCEPT):
        image_generator = torch.Generator(device=device).manual_seed(SEED + k * 1000 + j)
        image = generator.generate(h.to(dtype=torch.float16), generator=image_generator)
        path = os.path.join(OUTPUT_DIR, texts[k], f"{j}.png")
        os.makedirs(os.path.dirname(path), exist_ok=True)
        image.save(path)
        print("Image saved to", path)


## 10. 简单检查输出


In [ ]:
generated = []
for root, _, files in os.walk(OUTPUT_DIR):
    for name in files:
        if name.lower().endswith((".png", ".jpg", ".jpeg")):
            generated.append(os.path.join(root, name))
print("generated images:", len(generated))
print("output dir:", OUTPUT_DIR)
generated[:5]